# 14. Score degeneracy and a tie-aware transport panel

Follow-up to notebook 13, which found that switching the threshold from the linear-interpolated benign quantile to the order statistic moves FNR by 0.2 to 0.5 in several cells. That size of change across a one-benign-score gap means attack scores are piled in a razor-thin band at the threshold. This notebook measures that directly and re-reports the panel under a declared, tie-aware protocol. Cached scores only, no GPU.

**Outputs**
- `reports/score_degeneracy.csv`: per detector x shift, the threshold under five definitions, FNR and FPR under each, the attack and benign mass in the interpolation band and exactly at the threshold, unique-score counts and modal mass.
- `reports/panel_tie_aware.csv`: the transport panel re-reported at the guaranteed-FPR threshold with both tie conventions and the FNR spread.
- `figures/score_degeneracy.png`: ECDFs of attack, calibration-benign and target-benign scores near the thresholds, log scale.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True)
print('Session ready.')

## Stage A. Load frames and cached frozen scores (as notebook 13)

In [ ]:
import importlib, numpy as np, pandas as pd, os
import data_loaders, detectors
for m in (data_loaders, detectors): importlib.reload(m)
from data_loaders import (load_deepset, load_bipia_categorized, load_notinject, load_jailbreak,
                          BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS)
from detectors import threshold_at_fpr

deepset = load_deepset()
bip     = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
noti    = load_notinject()
jb      = load_jailbreak()
FRAMES  = {'deepset': deepset, 'bipia_cat': bip, 'notinject': noti, 'jailbreak': jb}

TAGS = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
P, missing = {}, []
for tag in TAGS:
    P[tag] = {}
    for key, f in FRAMES.items():
        fp = f'data/score_{tag}_{key}.npy'
        if not os.path.exists(fp): missing.append(fp); continue
        p = np.load(fp); assert len(p) == len(f), f'{fp}: {len(p)} vs {len(f)} rows'
        P[tag][key] = p
if missing: raise SystemExit('Missing score caches:\n  ' + '\n  '.join(missing))

def cells_for(tag):
    f = FRAMES; p = P[tag]
    ben_src = p['deepset'][f['deepset'].label.values == 0]
    bipben = f['bipia_cat'].label.values == 0
    H = f['bipia_cat'].meta.isin(BIPIA_HARMFUL).values
    J = f['bipia_cat'].meta.isin(BIPIA_HIJACK).values
    cells = {}
    cells['direct'] = (f['deepset'].label.values, p['deepset'])
    cells['indirect_harmful'] = (np.r_[np.ones(H.sum()), np.zeros(bipben.sum())], np.r_[p['bipia_cat'][H], p['bipia_cat'][bipben]])
    cells['indirect_hijack']  = (np.r_[np.ones(J.sum()), np.zeros(bipben.sum())], np.r_[p['bipia_cat'][J], p['bipia_cat'][bipben]])
    cells['jailbreak'] = (f['jailbreak'].label.values, p['jailbreak'])
    return cells, ben_src

SHIFTS = ['direct', 'indirect_harmful', 'indirect_hijack', 'jailbreak']
CELLS = {tag: cells_for(tag) for tag in TAGS}
print('loaded:', {tag: {k: int((y==1).sum()) for k,(y,p) in CELLS[tag][0].items()} for tag in TAGS})

## Stage B. Diagnostics module

Five thresholds on the calibration benigns at a 1 percent false-positive target: the published linear-interpolated quantile, the `higher`, `lower` and `nearest` order statistics, and `t_guar`, the smallest benign score at which the benign false-positive rate is guaranteed at or below 1 percent. The decision rule everywhere is block if p is at least t, so a miss is p below t. `FNR ties as miss` counts an attack tied exactly with the threshold as a miss, which is the security-conservative reading when the threshold sits on a mass point.

In [ ]:
%%writefile src/score_diagnostics.py
"""Score-degeneracy diagnostics and tie-aware thresholds for the transport panel."""
from __future__ import annotations
import numpy as np


def unique_stats(p):
    p = np.asarray(p, float)
    vals, counts = np.unique(p, return_counts=True)
    i = int(np.argmax(counts))
    return dict(n=int(len(p)), n_unique=int(len(vals)), modal_value=float(vals[i]),
                modal_mass=float(counts[i] / len(p)), min_value=float(vals[0]),
                mass_at_min=float(counts[0] / len(p)))


def thresholds(p_ben, fpr=0.01):
    """Five threshold definitions on the calibration benigns for a target FPR."""
    p_ben = np.asarray(p_ben, float)
    q = 1.0 - fpr
    T = {m: float(np.quantile(p_ben, q, method=m)) for m in ("linear", "higher", "lower", "nearest")}
    cand = np.unique(p_ben)
    ok = [s for s in cand if (p_ben >= s).mean() <= fpr]      # block iff p >= s guarantees FPR <= fpr
    T["guar"] = float(min(ok)) if ok else float(cand[-1])
    return {f"t_{k}": v for k, v in T.items()}


def cell_report(p_atk, p_ben_cal, p_ben_tgt=None, fpr=0.01, min_misses=10):
    p_atk = np.asarray(p_atk, float); p_ben_cal = np.asarray(p_ben_cal, float)
    T = thresholds(p_ben_cal, fpr); out = dict(T)
    for k, t in T.items():
        out[f"FNR@{k}"] = float((p_atk < t).mean())
        out[f"calFPR@{k}"] = float((p_ben_cal >= t).mean())
    tg = T["t_guar"]
    out["FNR@t_guar_ties_as_miss"] = float((p_atk <= tg).mean())
    out["n_atk_eq_t_guar"] = int((p_atk == tg).sum())
    out["n_ben_eq_t_guar"] = int((p_ben_cal == tg).sum())
    lo, hi = sorted([T["t_linear"], T["t_higher"]])
    out["n_atk_in_interp_band"] = int(((p_atk >= lo) & (p_atk <= hi)).sum())
    out["frac_atk_in_interp_band"] = out["n_atk_in_interp_band"] / len(p_atk)
    fnrs = [out[f"FNR@{k}"] for k in T] + [out["FNR@t_guar_ties_as_miss"]]
    out["FNR_spread"] = float(max(fnrs) - min(fnrs))
    if p_ben_tgt is not None and len(p_ben_tgt):
        p_ben_tgt = np.asarray(p_ben_tgt, float)
        out["tgtFPR@t_guar"] = float((p_ben_tgt >= tg).mean())
        out["tgtFPR@t_linear"] = float((p_ben_tgt >= T["t_linear"]).mean())
    miss = p_atk < tg
    out["S@t_guar"] = float(np.mean(1 - p_atk[miss])) if miss.sum() >= min_misses else np.nan
    for pre, arr in (("atk", p_atk), ("calben", p_ben_cal)):
        out.update({f"{pre}_{k}": v for k, v in unique_stats(arr).items()})
    # how many distinct attack score values fall strictly below the guaranteed threshold
    out["atk_n_unique_below_t_guar"] = int(len(np.unique(p_atk[p_atk < tg])))
    return out


## Stage C. Run the diagnostics on every cell

In [ ]:
import importlib, score_diagnostics; importlib.reload(score_diagnostics)
from score_diagnostics import cell_report, unique_stats

rows = []
for tag in TAGS:
    cells, ben_src = CELLS[tag]
    for sh in SHIFTS:
        y, p = cells[sh]; pa = p[y == 1]; pb = p[y == 0]
        r = cell_report(pa, ben_src, p_ben_tgt=(pb if sh != 'direct' else None))
        rows.append(dict(detector=LABEL[tag], shift=sh, n_atk=int(len(pa)), **r))
DG = pd.DataFrame(rows)
DG.to_csv('reports/score_degeneracy.csv', index=False)
pd.set_option('display.width', 250)

print('=== thresholds on the calibration benigns (per detector) ===')
print(DG[DG['shift']=='direct'][['detector','t_linear','t_lower','t_nearest','t_higher','t_guar','calFPR@t_linear','calFPR@t_guar']].round(5).to_string(index=False))

print('\n=== FNR under each threshold definition, and the spread ===')
print(DG[['detector','shift','FNR@t_linear','FNR@t_lower','FNR@t_nearest','FNR@t_higher','FNR@t_guar','FNR@t_guar_ties_as_miss','FNR_spread']].round(3).to_string(index=False))

print('\n=== mass at the threshold: attacks in the interpolation band, attacks and benigns exactly at t_guar ===')
print(DG[['detector','shift','n_atk','n_atk_in_interp_band','frac_atk_in_interp_band','n_atk_eq_t_guar','n_ben_eq_t_guar']].round(3).to_string(index=False))

print('\n=== score degeneracy: unique values, modal mass, saturation at the minimum ===')
print(DG[['detector','shift','atk_n','atk_n_unique','atk_modal_value','atk_modal_mass','atk_n_unique_below_t_guar','calben_n_unique','calben_modal_value','calben_modal_mass','calben_mass_at_min']].round(5).to_string(index=False))

## Stage D. Tie-aware panel

The panel re-reported at `t_guar` (block if p at least t, benign FPR guaranteed at or below 1 percent on the calibration set), with both tie conventions, the target-cell benign FPR, S, and the FNR spread across all definitions. This is the table that should replace the published Table 2 if the spread is material.

In [ ]:
TA = DG[['detector','shift','n_atk','t_guar','FNR@t_guar','FNR@t_guar_ties_as_miss','n_atk_eq_t_guar','calFPR@t_guar','tgtFPR@t_guar','S@t_guar','FNR@t_linear','FNR_spread']].copy()
TA = TA.rename(columns={'FNR@t_guar':'FNR','FNR@t_guar_ties_as_miss':'FNR_ties_as_miss','FNR@t_linear':'FNR_published_linear'})
TA.to_csv('reports/panel_tie_aware.csv', index=False)
print(TA.round(4).to_string(index=False))

print('\n=== headline: max FNR spread per detector, and cells with any attack exactly on the threshold ===')
print(DG.groupby('detector').FNR_spread.max().round(3).to_string())
on_t = DG[DG.n_atk_eq_t_guar > 0][['detector','shift','n_atk_eq_t_guar','n_ben_eq_t_guar']]
print('\ncells with attacks tied exactly at t_guar:'); print('NONE' if on_t.empty else on_t.to_string(index=False))
big = DG[DG.FNR_spread >= 0.05][['detector','shift','FNR_spread','frac_atk_in_interp_band']]
print('\ncells where FNR moves by 0.05 or more with the threshold definition:'); print('NONE' if big.empty else big.round(3).to_string(index=False))

## Stage E. Figure: where the attacks sit relative to the thresholds

In [ ]:
import matplotlib.pyplot as plt
def ecdf_xy(v):
    v = np.sort(np.clip(np.asarray(v, float), 1e-6, 1)); return v, np.arange(1, len(v)+1) / len(v)
fig, axes = plt.subplots(len(TAGS), 2, figsize=(12, 3.4*len(TAGS)))
for i, tag in enumerate(TAGS):
    cells, ben_src = CELLS[tag]
    for j, sh in enumerate(['indirect_harmful', 'indirect_hijack']):
        ax = axes[i, j]; y, p = cells[sh]; pa, pb = p[y==1], p[y==0]
        for v, lab, c in ((pa, 'attacks', 'C3'), (ben_src, 'calibration benigns (deepset)', 'C0'), (pb, 'target benigns (BIPIA hosts)', 'C2')):
            x, F = ecdf_xy(v); ax.step(x, F, where='post', label=lab, color=c)
        row = DG[(DG.detector==LABEL[tag]) & (DG['shift']==sh)].iloc[0]
        for k, ls in (('t_linear', ':'), ('t_higher', '--'), ('t_guar', '-')):
            ax.axvline(row[k], color='k', ls=ls, lw=1, label=k)
        ax.set_xscale('log'); ax.set_xlim(1e-6, 1); ax.set_ylim(0, 1)
        ax.set_title(f'{LABEL[tag]}: {sh}', fontsize=10); ax.set_xlabel('attack-probability p (log)'); ax.set_ylabel('ECDF')
        ax.grid(alpha=.3); ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.savefig('figures/score_degeneracy.png', dpi=150); plt.show()

## Stage E2. Regenerate notebook 13's deferral figure

`figures/deferral_bound.png` from notebook 13 was blank because its selector used `.shift` attribute access, which pandas resolves to the `shift()` method. Rebuilt here from the committed CSV.

In [ ]:
DEF = pd.read_csv('reports/deferral_bound.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for det in DEF['detector'].unique():
    d = DEF[(DEF['detector'] == det) & (DEF['shift'] == 'indirect_hijack')].sort_values('budget')
    axes[0].plot(d['budget'], d['exploit_rate_lower_bound'], marker='o', label=det)
    axes[1].plot(d['budget'], d['misses_recovered_by_abstention'], marker='o', label=det)
axes[0].set_xscale('log'); axes[0].set_xlabel('deferral budget b'); axes[0].set_ylabel('exploit-rate lower bound'); axes[0].set_title('Deferral bound, indirect-hijack')
axes[1].set_xscale('log'); axes[1].set_xlabel('deferral budget b'); axes[1].set_ylabel('fraction of misses recovered'); axes[1].set_title('Threshold-keyed abstention, indirect-hijack')
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('figures/deferral_bound.png', dpi=150); plt.show()
print('deferral_bound.png regenerated with', DEF['detector'].nunique(), 'detectors')

## Stage F. Log and commit

In [ ]:
from reslog import log_result
summary = ('Tie-aware panel:\n' + TA.round(4).to_string(index=False)
           + '\n\nMax FNR spread per detector:\n' + DG.groupby('detector').FNR_spread.max().round(3).to_string()
           + '\n\nDegeneracy (attacks):\n' + DG[['detector','shift','atk_n_unique','atk_modal_mass','frac_atk_in_interp_band','n_atk_eq_t_guar']].round(3).to_string(index=False))
log_result('nb14: score degeneracy and tie-aware transport panel', summary, csv_df=TA, csv_name='panel_tie_aware.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb14: score-degeneracy diagnostics and tie-aware transport panel; add src/score_diagnostics.py; regenerate deferral_bound.png (nb13 figure was blank)"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)